# TAQ 1-minute prices — yearly parallel pull (CSV.gz) — `bis`

Variant of `1-download_hf_prices_yearly.ipynb` with proven optimizations from diagnostic on 2020-06-15.

## Changes vs v1

**SQL filter — sargable NULL-aware split** (~4.87× faster server-side, validated by `EXPLAIN ANALYZE`)

The original filter `(sym_root, COALESCE(sym_suffix, '')) IN (...)` defeats the columnar scan's pruning on `sym_suffix` because of the `COALESCE` wrapper. The bis filter splits the predicate:

```sql
(sym_suffix IS NULL AND sym_root IN (...)) OR
(sym_suffix IS NOT NULL AND (sym_root, sym_suffix) IN (...))
```

→ each branch is sargable; the columnar scan can prune chunks via min/max stats on `sym_suffix`.

**Median dedup on microsecond ties** (determinism fix)

When two trades share the exact same `time_m`, the original `merge_asof(direction='backward')` picks one based on row order in the DataFrame — which depends on the Postgres execution plan, hence non-deterministic across runs. The bis pipeline aggregates trades on `(sym_root, sym_suffix, time_m)` by **median** before fill, matching the pre-2015 path. ~50 cents-level diffs/day eliminated.

**Improved logging**
- Per-day breakdown (DEBUG): `rows | SQL time | Python time | total`
- Periodic ETA (INFO): every 25 days, year report `done/total | elapsed | speed | ETA`
- Sanity warning: days returning < 50 K rows are flagged (early close / partial data)
- Errors surfaced in console (`WARNING+` → stderr) in addition to file
- Final summary: `logs/hf_prices_bis_<ts>_summary.json` — years, status, days, rows, durations, errors

**Expected wall time** (5 workers, 1 107 PERMNO, 2018-2025): ~2-3 h vs ~17 h with v1 SQL.

In [ ]:
import wrds
import pandas as pd
import numpy as np
import gzip
import json
import sys
import time
import logging
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

## 0) WRDS connection (main thread)

We open a primary connection here for the one-shot mapping pull below. Each worker thread will open its **own** connection (psycopg2 is not safe to share across threads).

In [ ]:
WRDS_USERNAME = 'aadmberrada'
db = wrds.Connection(wrds_username=WRDS_USERNAME)

## Configuration

In [ ]:
FIRST_DATE = '2021-01-01'
LAST_DATE  = '2025-12-31'

# Universe — full S&P 500 from CRSP CIZ panel
univ_df = pd.read_parquet('../data/wrds/sp500_constituents_daily.parquet')
PERMNO_LIST = univ_df['permno'].dropna().astype('int32').unique().tolist()

OUT_DIR = Path('../data/wrds/high_freq_prices')
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS       = list(range(int(FIRST_DATE[:4]), int(LAST_DATE[:4]) + 1))
MAX_WORKERS = 5   # 5 SQL connections (WRDS soft cap, fits 12 GB free RAM)

print(f'Universe: {len(PERMNO_LIST):,} PERMNO  |  Years: {YEARS}  |  Workers: {MAX_WORKERS}')
print(f'Output:   {OUT_DIR}')

## 1) PERMNO → sym_root/sym_suffix mapping (TAQMCLINK)

Pulled once on the main thread; the result is read-only and shared across workers.

In [ ]:
permno_tuple = tuple(int(p) for p in PERMNO_LIST)

taq_map = db.raw_sql(f"""
    SELECT DISTINCT permno,
                    sym_root,
                    COALESCE(sym_suffix, '') AS sym_suffix
    FROM wrdsapps.taqmclink
    WHERE permno IN {permno_tuple}
      AND sym_root IS NOT NULL
""")

taq_map = taq_map.astype({
    'permno':     'int32',
    'sym_root':   'category',
    'sym_suffix': 'category',
})

print(f"Universe PERMNO: {len(permno_tuple):,}")
print(f"(root, suffix) pairs: {len(taq_map):,}")
taq_map.head()

## 2) Per-day extraction functions (thread-safe, with bis patches)

All functions accept `db` as the first argument — no module-level connection.

Server-side filters:
- RTH only: `09:30:00 ≤ time_m ≤ 16:00:00`
- Real trade: `price > 0`
- Major exchanges: `ex IN ('N','P','T','Q','A','D')`
- No correction: `tr_corr = '00'`
- Sale conditions: blank or letters E/F
- **Universe filter (bis)**: NULL-aware split on `sym_suffix` for sargable pruning

Median dedup on `(sym_root, sym_suffix, time_m)` before previous-tick fill — eliminates non-determinism on microsecond ties.

In [ ]:
MEDIAN_CUTOFF = pd.Timestamp('2015-04-01')
GRID_TD       = pd.timedelta_range(start='09:30:00', end='16:00:00', freq='1min')   # 391 entries


def build_symbol_filter(taq_map_local):
    """Sargable NULL-aware (sym_root, sym_suffix) WHERE clause.

    Splits NULL/non-NULL suffix branches so the columnar scan can prune
    chunks via min/max statistics on sym_suffix. Versus a single
    COALESCE-wrapped IN list, measured ~4.87x faster on TAQ MSEC ctm
    tables (EXPLAIN ANALYZE on 2020-06-15: 159.7s -> 32.8s).

    Args:
        taq_map_local: DataFrame with columns sym_root, sym_suffix
            (sym_suffix may be empty string for symbols without suffix).

    Returns:
        SQL fragment to drop into the WHERE clause, e.g.
        ``(sym_suffix IS NULL AND sym_root IN ('AAPL', ...)) OR
          (sym_suffix IS NOT NULL AND (sym_root, sym_suffix) IN (('BRK','B'), ...))``
    """
    df = taq_map_local[['sym_root', 'sym_suffix']].astype(str).drop_duplicates()
    null_mask = (df['sym_suffix'] == '')
    roots_null = sorted(df.loc[null_mask, 'sym_root'].unique().tolist())
    pairs_nn   = list(df.loc[~null_mask].itertuples(index=False))
    parts = []
    if roots_null:
        roots_sql = ', '.join(f"'{r}'" for r in roots_null)
        parts.append(f"(sym_suffix IS NULL AND sym_root IN ({roots_sql}))")
    if pairs_nn:
        pairs_sql = ', '.join(f"('{r}','{s}')" for r, s in pairs_nn)
        parts.append(f"(sym_suffix IS NOT NULL AND (sym_root, sym_suffix) IN ({pairs_sql}))")
    return ' OR '.join(parts)


SYMBOL_FILTER = build_symbol_filter(taq_map)
print(f"Symbol filter built: {len(SYMBOL_FILTER):,} chars")


def fetch_day(db, yyyymmdd, symbol_filter):
    """Server-side filtered pull of one day's trades.

    Returns a DataFrame with columns date, time_m (Timedelta), sym_root
    (category), sym_suffix (category, '' when missing), price (float64).
    Empty DataFrame if no trade survived the filters.
    """
    df = db.raw_sql(f"""
        SELECT date,
               time_m,
               sym_root,
               COALESCE(sym_suffix, '') AS sym_suffix,
               price
        FROM taqmsec.ctm_{yyyymmdd}
        WHERE time_m BETWEEN TIME '09:30:00' AND TIME '16:00:00'
          AND price > 0
          AND ex IN ('N','P','T','Q','A','D')
          AND tr_corr = '00'
          AND (tr_scond IS NULL OR tr_scond = '' OR tr_scond !~ '[ABCDGHIJKLMNOPQRSTUVWXYZ]')
          AND ({symbol_filter})
    """, date_cols=['date'])
    if not df.empty:
        df['time_m']     = pd.to_timedelta(df['time_m'].astype(str))
        df['sym_root']   = df['sym_root'].astype('category')
        df['sym_suffix'] = df['sym_suffix'].astype('category')
        df['price']      = df['price'].astype('float64')
    return df


def previous_tick_minute_grid(trades, day):
    """Reindex trades onto a 1-min grid 09:30..16:00 with previous-tick fill.

    Trades sharing the same (sym_root, sym_suffix, time_m) are aggregated
    by median first. This makes the output deterministic on microsecond
    ties (TAQ MSEC has multiple trades printed at the same microsecond on
    liquid names; row order from Postgres depends on the execution plan).
    Without this dedup, two runs of the same query produce ~50 cents-level
    price diffs per day per ticker.

    Args:
        trades: DataFrame from `fetch_day` — columns sym_root, sym_suffix,
            time_m (Timedelta), price.
        day: Timestamp of the trading day (for the date column).

    Returns:
        DataFrame with one row per (ticker, minute) on the 391-min grid;
        price filled with the previous trade if no tick at that minute,
        NaN if no trade up to that point in the day.
    """
    if trades.empty:
        return trades.iloc[0:0]
    out_chunks = []
    for (root, suffix), g in trades.groupby(['sym_root', 'sym_suffix'],
                                            sort=False, observed=True):
        # Determinism: median over microsecond ties before fill
        g = (g.groupby('time_m', as_index=False)['price'].median()
               .sort_values('time_m'))
        left = pd.DataFrame({'time_m': GRID_TD})
        merged = pd.merge_asof(left, g, on='time_m', direction='backward')
        merged['date']       = day.date()
        merged['sym_root']   = root
        merged['sym_suffix'] = suffix
        merged['time'] = (pd.Timestamp('1900-01-01') + merged['time_m']).dt.strftime('%H:%M')
        out_chunks.append(merged[['date', 'time', 'sym_root', 'sym_suffix', 'price']])
    return pd.concat(out_chunks, ignore_index=True)


def process_one_day(db, yyyymmdd, taq_map_local, symbol_filter):
    """Pull + fill one day. Returns (df, sql_time_s, python_time_s).

    The split timing is exposed so the caller can log SQL vs Python
    cost separately — useful for diagnosing whether a slowdown comes
    from the WRDS server or from local processing.
    """
    day = pd.to_datetime(yyyymmdd, format='%Y%m%d')

    t0 = time.perf_counter()
    raw = fetch_day(db, yyyymmdd, symbol_filter)
    if day < MEDIAN_CUTOFF and not raw.empty:
        # Pre-2015: TAQ has duplicate prints / tick differences; collapse
        # to a median per (ticker, time_m) before the previous-tick fill.
        raw = (raw.groupby(['sym_root', 'sym_suffix', 'time_m'],
                           as_index=False, observed=True)
                  .agg(date=('date', 'first'), price=('price', 'median')))
    t_sql = time.perf_counter() - t0

    t0 = time.perf_counter()
    minute_grid = previous_tick_minute_grid(raw, day)
    if minute_grid.empty:
        return minute_grid, t_sql, time.perf_counter() - t0

    minute_grid = minute_grid.merge(taq_map_local,
                                    on=['sym_root', 'sym_suffix'], how='left')
    minute_grid['ticker'] = (minute_grid['sym_root'].astype(str)
                             + minute_grid['sym_suffix'].astype(str))
    minute_grid['permno'] = minute_grid['permno'].astype('Int32')
    minute_grid['price']  = minute_grid['price'].round(4)
    out = minute_grid[['date', 'time', 'permno', 'ticker', 'price']]
    t_py = time.perf_counter() - t0
    return out, t_sql, t_py

## 3) Year worker + parallel runner with rich logs

**Console**: `tqdm` years bar + `WARNING+` from logger (errors and partial-day flags).

**File** (`logs/hf_prices_bis_<ts>.log`):
- DEBUG line per day: `rows | SQL time | Python time | total`
- INFO line every 25 days: `done/total | elapsed | speed | ETA`

**JSON summary** (`logs/hf_prices_bis_<ts>_summary.json`): final report — config + per-year status, days, rows, durations, errors. Reproducible and machine-readable.

Tail the file in another terminal for live progress:
```
tail -f logs/hf_prices_bis_*.log
```

In [ ]:
# ---- Logging setup ----
log_dir = Path('../logs')
log_dir.mkdir(parents=True, exist_ok=True)
ts = time.strftime('%Y%m%d_%H%M%S')
log_path     = log_dir / f"hf_prices_bis_{ts}.log"
summary_path = log_dir / f"hf_prices_bis_{ts}_summary.json"

logger = logging.getLogger('hf_prices_bis')
logger.setLevel(logging.DEBUG)
logger.propagate = False
for h in list(logger.handlers):  # avoid duplicate handlers on cell re-run
    logger.removeHandler(h)

# File handler: DEBUG (verbose, full per-day timeline)
fh_log = logging.FileHandler(log_path, encoding='utf-8')
fh_log.setLevel(logging.DEBUG)
fh_log.setFormatter(logging.Formatter(
    '%(asctime)s [%(threadName)s] %(levelname)-7s %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
))
logger.addHandler(fh_log)

# Console handler: WARNING+ only (notebook stays clean except on errors)
ch_log = logging.StreamHandler(sys.stderr)
ch_log.setLevel(logging.WARNING)
ch_log.setFormatter(logging.Formatter('%(levelname)s [%(threadName)s] %(message)s'))
logger.addHandler(ch_log)

logger.info(
    f'=== HF prices bis pull starting | years={YEARS} '
    f'| universe={len(PERMNO_LIST)} | workers={MAX_WORKERS} ==='
)
print(f'Log:     {log_path}')
print(f'Summary: {summary_path}\n')


# ---- Year worker ----
ETA_EVERY_N_DAYS  = 25      # log progress + ETA every N days
PARTIAL_DAY_ROWS  = 50_000  # warn if a day returns fewer rows than this

def process_year(year):
    out_path = OUT_DIR / f'hf_prices_{year}.csv.gz'
    if out_path.exists():
        logger.info(f'year {year}: SKIP (already exists)')
        return dict(year=year, status='skipped', path=str(out_path),
                    rows=0, days=0, t_sql_s=0.0, t_py_s=0.0,
                    elapsed_s=0.0, error=None)

    tmp_path = out_path.with_suffix(out_path.suffix + '.tmp')
    db_local = wrds.Connection(wrds_username=WRDS_USERNAME)
    try:
        ctm = db_local.raw_sql(f"""
            SELECT SUBSTRING(table_name FROM 5 FOR 8) AS yyyymmdd
            FROM information_schema.tables
            WHERE table_schema = 'taqmsec'
              AND table_name ~ '^ctm_[0-9]{{8}}$'
              AND SUBSTRING(table_name FROM 5 FOR 8) BETWEEN '{year}0101' AND '{year}1231'
            ORDER BY table_name
        """)
        if ctm.empty:
            logger.warning(f'year {year}: no CTM tables in range')
            return dict(year=year, status='no_tables', path=None,
                        rows=0, days=0, t_sql_s=0.0, t_py_s=0.0,
                        elapsed_s=0.0, error=None)

        n_days_total = len(ctm)
        logger.info(f'year {year}: starting {n_days_total} days')

        n_rows, n_days = 0, 0
        sum_t_sql, sum_t_py = 0.0, 0.0
        t0_y = time.perf_counter()

        with gzip.open(tmp_path, 'wt', encoding='utf-8', newline='') as fhgz:
            first = True
            for yyyymmdd in ctm['yyyymmdd']:
                # Per-day try/except so one bad day does not kill the year
                try:
                    day_df, t_sql, t_py = process_one_day(
                        db_local, yyyymmdd, taq_map, SYMBOL_FILTER
                    )
                except Exception as e:
                    logger.warning(
                        f'year {year} day {yyyymmdd}: SKIPPED '
                        f'({type(e).__name__}: {e})'
                    )
                    continue

                sum_t_sql += t_sql
                sum_t_py  += t_py

                if day_df.empty:
                    logger.debug(f'year {year} day {yyyymmdd}: empty after filters')
                    continue

                if len(day_df) < PARTIAL_DAY_ROWS:
                    logger.warning(
                        f'year {year} day {yyyymmdd}: only {len(day_df):,} rows '
                        f'(< {PARTIAL_DAY_ROWS:,}) — partial / early-close day?'
                    )

                day_df.to_csv(fhgz, header=first, index=False,
                              float_format='%.4f', mode='a')
                first   = False
                n_rows += len(day_df)
                n_days += 1

                logger.debug(
                    f'year {year} day {yyyymmdd}: {len(day_df):>8,} rows | '
                    f'SQL {t_sql:5.1f}s | Python {t_py:4.1f}s | '
                    f'total {t_sql + t_py:5.1f}s'
                )

                # Periodic ETA
                if n_days > 0 and n_days % ETA_EVERY_N_DAYS == 0:
                    elapsed   = time.perf_counter() - t0_y
                    avg       = elapsed / n_days
                    remaining = n_days_total - n_days
                    eta       = remaining * avg
                    logger.info(
                        f'year {year}: {n_days:>3}/{n_days_total} days done '
                        f'in {elapsed/60:5.1f} min  '
                        f'({avg:4.1f}s/day, ETA {eta/60:5.1f} min)'
                    )

        tmp_path.rename(out_path)
        elapsed = time.perf_counter() - t0_y
        logger.info(
            f'year {year}: DONE {n_days} days, {n_rows:,} rows in '
            f'{elapsed/60:.1f} min '
            f'(SQL {sum_t_sql/60:.1f} min, Python {sum_t_py/60:.1f} min)'
        )
        return dict(year=year, status='ok', path=str(out_path),
                    rows=n_rows, days=n_days,
                    t_sql_s=sum_t_sql, t_py_s=sum_t_py,
                    elapsed_s=elapsed, error=None)

    except Exception as e:
        if tmp_path.exists():
            tmp_path.unlink()
        logger.exception(f'year {year}: failed')
        return dict(year=year, status='error', path=None,
                    rows=0, days=0, t_sql_s=0.0, t_py_s=0.0,
                    elapsed_s=0.0, error=f'{type(e).__name__}: {e}')
    finally:
        db_local.close()


# ---- Parallel runner ----
t0_total = time.perf_counter()
results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(process_year, y): y for y in YEARS}
    with tqdm(total=len(futures), desc='Years', unit='yr') as pbar:
        for fut in as_completed(futures):
            r = fut.result()
            results.append(r)
            pbar.write(
                f"  year {r['year']}: {r['days']:>4} days, "
                f"{r['rows']:>11,} rows  "
                f"[{r['elapsed_s']/60:5.1f} min]  "
                f"SQL {r['t_sql_s']/60:5.1f} min | Python {r['t_py_s']/60:5.1f} min  "
                f"-> {r['status']}"
            )
            pbar.update(1)

elapsed_total = time.perf_counter() - t0_total
logger.info(f'=== HF prices bis pull complete in {elapsed_total/60:.1f} min ===')

# ---- Summary JSON ----
summary = {
    'started_at':    datetime.fromtimestamp(t0_total).isoformat(),
    'finished_at':   datetime.now().isoformat(),
    'wall_time_sec': round(elapsed_total, 1),
    'wall_time_min': round(elapsed_total / 60, 1),
    'config': {
        'first_date':  FIRST_DATE,
        'last_date':   LAST_DATE,
        'universe_n':  len(PERMNO_LIST),
        'years':       YEARS,
        'max_workers': MAX_WORKERS,
    },
    'years': sorted(results, key=lambda r: r['year']),
}
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, default=str)

print(f'\nDone in {elapsed_total/60:.1f} min')
print(f'Log:     {log_path}')
print(f'Summary: {summary_path}\n')
print(f'Files in {OUT_DIR}:')
for p in sorted(OUT_DIR.glob('hf_prices_*.csv.gz')):
    print(f'  {p.name:30s}  {p.stat().st_size/1e6:>8.1f} MB')

## 4) Sanity checks (read multi-year CSV.gz back)

In [ ]:
files = sorted(OUT_DIR.glob('hf_prices_*.csv.gz'))
if not files:
    print('No output files found.')
else:
    read_dtypes = {
        'permno': 'Int32',
        'ticker': 'category',
        'price':  'float64',
    }
    hf_prices = pd.concat(
        [pd.read_csv(f, parse_dates=['date'], dtype=read_dtypes) for f in files],
        ignore_index=True,
    )
    print(f'Loaded {len(files)} year file(s): {len(hf_prices):,} rows total\n')

    print(f"Memory usage:   {hf_prices.memory_usage(deep=True).sum()/1e6:.1f} MB")
    print(f"Unique days:    {hf_prices['date'].nunique():>4}")
    print(f"Unique PERMNO:  {hf_prices['permno'].nunique():>4}")
    print(f"Unique tickers: {hf_prices['ticker'].nunique():>4}")
    n_nan = hf_prices['price'].isna().sum()
    print(f"Price NaN:      {n_nan:>10,}  ({n_nan/len(hf_prices)*100:.2f}%)")

    cnt = hf_prices.groupby(['date', 'permno'], observed=True).size()
    print(f"\nMinute counts per (date, permno) — should be 391:")
    print(f"  min={cnt.min()}  median={int(cnt.median())}  max={cnt.max()}")

    print(f'\nFirst 5 rows:')
    print(hf_prices.head().to_string(index=False))

## 5) Disconnect

In [ ]:
db.close()